# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalalostcode/FlyrankAI_ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook converts the validated W06 ranking experiment into a human-reviewed editorial queue. It uses the conservative \`B2_current_rank_blend\`, not the marginally higher development candidate: B2 measured **92.4% ± 3.3% mean P@50** across five client-grouped folds, while A11 measured 92.8% ± 2.3% (+0.4 percentage points) on the same development folds.

The output is **current-snapshot, cross-client decision support**. It is not a future forecast, causal estimate, or instruction to edit content automatically.

## 1. Ranked actions + reason codes

The queue ranks up to 50 items inside each pseudonymous client group. Ranks 1–10 are \`review_now\`, 11–25 are \`review_next\`, and 26–50 are \`monitor\`. The score is a relative rank signal—not a calibrated probability.

Reason codes translate observable fields into review prompts:

| Reason code | Observable trigger | Human review prompt |
|---|---|---|
| \`low_ctr_visible\` | ≥500 impressions, position 1–20, CTR <0.5% | Inspect query intent, SERP features, title, and description |
| \`thin_visible\` | ≥250 impressions and 1–1,199 words | Check coverage and usefulness; do not pad for length |
| \`stale_visible\` | ≥500 impressions and ≥180 days since update | Verify whether facts, examples, and references are stale |
| \`page_one_aging\` | position 1–10 and content age ≥180 days | Protect a visible asset; inspect before changing it |
| \`weak_engagement_visible\` | ≥30 sessions and an observed rate between 0% and 30% | Check intent match, UX, and measurement quality |
| \`model_pattern_needs_diagnosis\` | High model rank without a threshold code | Diagnose manually; the model score is not an explanation |

The suggested action is deliberately phrased as an inspection or review. It does not assert why an item is declining.

In [1]:
from pathlib import Path
import importlib.util
import warnings

import pandas as pd

warnings.filterwarnings('ignore')

def locate_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'skills' / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the FlyRank repository.')

ROOT = locate_root()
MODULE_PATH = ROOT / 'work' / 'scripts' / 'action_playbook.py'
spec = importlib.util.spec_from_file_location('action_playbook', MODULE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f'Cannot load {MODULE_PATH}')
playbook = importlib.util.module_from_spec(spec)
spec.loader.exec_module(playbook)

work, experiment = playbook.load_inputs(ROOT)
queue = playbook.build_queue(work)
benchmark, safe_claim = playbook.validate_scope(experiment)
shadow = experiment['results'][playbook.SHADOW_MODEL]['summary']

print(f'Selected model: {playbook.SELECTED_MODEL} (conservative benchmark)')
print(
    f'Validation: mean P@50={benchmark["mean_p_at_50"]:.1%} ± '
    f'{benchmark["std_p_at_50"]:.1%}; '
    f'mean fold base rate={benchmark["mean_base_rate"]:.1%}'
)
print(
    f'Shadow candidate: {playbook.SHADOW_MODEL}, '
    f'mean P@50={shadow["mean_p_at_50"]:.1%} '
    f'({experiment["improvement_pp"]:+.1f}pp; not promoted)'
)
print(
    f'Operational queue: {len(queue):,} rows across '
    f'{queue["client_id"].nunique()} pseudonymous client groups'
)

preview = queue.sort_values('model_rank_score', ascending=False).head(12).copy()
preview['item'] = 'item_' + preview['content_id'].astype(str).str[-8:]
preview['group'] = 'group_' + preview['client_id'].astype(str).str[-6:]
print('\nTop queue preview (pseudonymous display keys; no outcome label):')
print(
    preview[
        ['item', 'group', 'client_rank', 'priority_tier', 'model_rank_score',
         'reason_codes', 'suggested_action']
    ].to_string(index=False)
)

Selected model: B2_current_rank_blend (conservative benchmark)
Validation: mean P@50=92.4% ± 3.3%; mean fold base rate=54.4%
Shadow candidate: A11_current_plus_top20_equal, mean P@50=92.8% (+0.4pp; not promoted)
Operational queue: 1,473 rows across 32 pseudonymous client groups

Top queue preview (pseudonymous display keys; no outcome label):
         item        group  client_rank priority_tier  model_rank_score    reason_codes       suggested_action
item_346533b1 group_a35f04            1    review_now          1.000000 low_ctr_visible inspect_search_snippet
item_8f7165e1 group_1e27de            1    review_now          0.999929 low_ctr_visible inspect_search_snippet
item_e82484db group_53d7e2            1    review_now          0.999826 low_ctr_visible inspect_search_snippet
item_e8ddc090 group_53d7e2            2    review_now          0.999826 low_ctr_visible inspect_search_snippet
item_7ec37ab8 group_fabef1            1    review_now          0.999826 low_ctr_visible inspect_sear

## 2. Intended use and limits

**User and decision.** An editorial or SEO analyst uses this queue to decide which pages deserve manual diagnosis first when review capacity is limited. Work the queue inside each client group; start with \`review_now\`, record the review outcome, and only then decide whether any edit is appropriate.

**Evidence carried by the score.** Each row has an out-of-fold score from a model that did not train on that row's client. Across the five fixed client-grouped folds, the selected blend measured 92.4% mean P@50 versus a 54.4% mean fold base rate. That supports a top-of-queue ranking claim for this snapshot.

**Limits.** The 30,000-row starter data is a single snapshot. Its threshold label is derived from current impression trend, and inherited 90-day predictors overlap the current label period. Therefore this playbook does not establish future forecasting, intervention impact, ROI, or a reason for decline. Performance may differ for new time periods, clients, data pipelines, and editorial capacities other than K=20/50/100. The 92.8% shadow candidate remains unpromoted because its +0.4pp gain was selected on the same development folds and did not reach the predeclared +5pp target.

In [2]:
metric_table = pd.DataFrame([
    {'measure': 'mean fold base rate', 'value': benchmark['mean_base_rate']},
    {'measure': 'mean precision@20', 'value': benchmark['mean_p_at_20']},
    {'measure': 'mean precision@50', 'value': benchmark['mean_p_at_50']},
    {'measure': 'std precision@50', 'value': benchmark['std_p_at_50']},
    {'measure': 'mean precision@100', 'value': benchmark['mean_p_at_100']},
    {'measure': 'mean ROC-AUC', 'value': benchmark['mean_roc_auc']},
    {'measure': 'mean average precision', 'value': benchmark['mean_average_precision']},
])
print(
    metric_table.assign(
        value=metric_table['value'].map(lambda value: f'{value:.1%}')
    ).to_string(index=False)
)
print('\nClaim approved by scope checks:')
print(safe_claim)

               measure value
   mean fold base rate 54.4%
     mean precision@20 92.0%
     mean precision@50 92.4%
      std precision@50  3.3%
    mean precision@100 88.6%
          mean ROC-AUC 64.3%
mean average precision 68.4%

Claim approved by scope checks:
On five client-grouped folds in this single-snapshot dataset, the selected model ranked the top 50 items at mean precision 92.4% ± 3.3%, compared with a 54.4% mean fold base rate. The queue is decision support for manual review, not a future forecast.


## 3. Human review + the no-go list

Before acting, the reviewer must check the page's ownership/canonical status, the underlying time series, seasonality and campaign effects, indexing or migration incidents, current query intent and SERP layout, factual freshness, content usefulness, and any legal or brand constraints. A threshold reason code is a prompt, not an explanation.

**Never automate these from the queue alone:**

- editing, expanding, unpublishing, redirecting, or changing canonical tags;
- treating the rank score as a decline probability or a cause;
- using outcome labels, trend fields, client IDs, or product scores as model features;
- comparing or naming clients in public outputs;
- deploying to a new period or new population without shadow validation;
- evaluating staff, sending outreach, or promising traffic impact from the score.

The reviewer records \`act\`, \`defer\`, or \`no_action\`, a short evidence note, and the actual intervention separately. That feedback is not silently recycled as a label.

In [3]:
review_sample = playbook.validate_queue(queue)
review_sample['item'] = (
    'item_' + review_sample['content_id'].astype(str).str[-8:]
)
print('Human-review sample: two high-ranked items per proposed action')
print(
    review_sample[
        ['item', 'client_rank', 'priority_tier', 'reason_codes', 'suggested_action']
    ].to_string(index=False)
)
print(
    f'\nSafety checks passed: {len(playbook.EXPORT_COLUMNS)} operational fields; '
    'no target, trend, fold, or shadow score exported.'
)

Human-review sample: two high-ranked items per proposed action
         item  client_rank priority_tier                  reason_codes             suggested_action
item_bf8810fa            2    review_now model_pattern_needs_diagnosis       diagnose_before_action
item_5c7434ac            5    review_now model_pattern_needs_diagnosis       diagnose_before_action
item_346533b1            1    review_now               low_ctr_visible       inspect_search_snippet
item_8f7165e1            1    review_now               low_ctr_visible       inspect_search_snippet
item_27f5a770            3    review_now   thin_visible|page_one_aging   review_depth_and_relevance
item_57118a6c            1    review_now   thin_visible|page_one_aging   review_depth_and_relevance
item_d5ae5c41           12   review_next                page_one_aging   review_facts_and_freshness
item_f31ba9bb           32       monitor                page_one_aging   review_facts_and_freshness
item_592bbb17           33       moni

## 4. Monitoring / retrain triggers

Monitor each scoring cycle and evaluate performance only after labels mature. A single alert starts diagnosis; two consecutive performance alerts pause model-led ordering and return the team to a transparent rule/manual queue.

Retraining is not automatic. It requires a documented data cutoff, the same leakage guardrails, grouped validation, and—before any future-looking claim—multiple temporal forecast origins from the daily warehouse. A challenger replaces B2 only after a predeclared material gain on untouched data, not after repeated tuning on these five folds.

In [4]:
monitoring_plan, reference_missingness, p50_pause_floor = (
    playbook.make_monitoring_plan(work, benchmark)
)
print(monitoring_plan.to_string(index=False))
print('\nReference missingness rates:')
for feature, rate in reference_missingness.items():
    print(f'  {feature:<25} {rate:.1%}')

                    signal              check                                     alert                                        response
  schema / required fields  every scoring run     any missing or renamed required field       stop scoring and repair upstream contract
       feature missingness  every scoring run >5 percentage-point increase vs reference    investigate source; do not zero-fill blindly
       score or action mix  every scoring run            >15 percentage-point mix shift      review drift and thresholds before release
         mature-label P@50 each labeled cycle           <82.4% or >10pp below reference    diagnose; pause after two consecutive alerts
           label base rate each labeled cycle                >10 percentage-point shift report recalibrated context; revalidate ranking
new client / future period  before deployment     outside validated snapshot/population    shadow mode plus grouped/temporal evaluation

Reference missingness rates:
  impressions_90d 

## 5. Exports for the paper

The operational CSV contains pseudonymous keys, ranks, review prompts, and supporting observable fields; it excludes the outcome label and trend columns. The JSON and PNG contain aggregate receipts only and are safe to reuse in the capstone paper. Dataset-like CSVs remain uncommitted under the repository's data policy, while the aggregate JSON and figure are the reproducibility receipts.

In [5]:
queue_path, metrics_path, figure_path, receipts = playbook.export_artifacts(
    ROOT,
    work,
    queue,
    experiment,
    benchmark,
    reference_missingness,
    p50_pause_floor,
)
print('Verified paper exports:')
for path in (queue_path, metrics_path, figure_path):
    print(f'  {path.relative_to(ROOT)} ({path.stat().st_size:,} bytes)')
print(
    f'\nQueue rows: {receipts["queue_rows"]:,}; aggregate receipts contain '
    'no client identifiers or outcome labels.'
)

Verified paper exports:
  work\outputs\action_playbook_queue.csv (229,272 bytes)
  work\outputs\action_playbook_metrics.json (2,234 bytes)
  work\outputs\action_playbook_summary.png (78,727 bytes)

Queue rows: 1,473; aggregate receipts contain no client identifiers or outcome labels.


## Self-check

- [x] Every section above is filled—markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries appear anywhere
- [x] Claims use observed, measured, directional, and decision-support language
- [x] Operational export excludes labels, trend fields, fold IDs, and the shadow score
- [x] Human review, no-go actions, drift checks, and retrain triggers are explicit
- [ ] Committed to the repo under \`work/notebooks/\`, then submit the repo URL on the card